In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA


In [2]:
option_data = pd.read_csv('results/bsm_model_prices.csv')[['Date', 'Type', 'S0', 'K', 'T', 'r', 'sigma', 'Close Price', 'LTP']]
option_data

,Date,Type,S0,K,T,r,sigma,Close Price,LTP
0,2025-11-12,PE,25875.8,26500.0,34.0,0.1,0.074745,533.05,533.05
1,2025-11-12,CE,25875.8,25900.0,34.0,0.1,0.074745,468.90,468.90
2,2025-11-12,CE,25875.8,27450.0,34.0,0.1,0.074745,21.00,20.00
3,2025-11-12,CE,25875.8,26200.0,34.0,0.1,0.074745,286.75,286.75
4,2025-11-12,CE,25875.8,25800.0,34.0,0.1,0.074745,505.50,504.80
...,...,...,...,...,...,...,...,...,...
2748,2025-12-16,PE,25860.1,24850.0,0.0,0.1,0.078564,0.10,0.05
2749,2025-12-16,CE,25860.1,24550.0,0.0,0.1,0.078564,1342.00,1342.00
2750,2025-12-16,PE,25860.1,24250.0,0.0,0.1,0.078564,0.10,0.05
2751,2025-12-16,CE,25860.1,25400.0,0.0,0.1,0.078564,471.05,460.50


In [3]:
dates = option_data['Date'].unique()
dates

array(['2025-11-12', '2025-11-13', '2025-11-14', '2025-11-17',
       '2025-11-18', '2025-11-19', '2025-11-20', '2025-11-21',
       '2025-11-24', '2025-11-25', '2025-11-26', '2025-11-27',
       '2025-11-28', '2025-12-01', '2025-12-02', '2025-12-03',
       '2025-12-04', '2025-12-05', '2025-12-08', '2025-12-09',
       '2025-12-10', '2025-12-11', '2025-12-12', '2025-12-15',
       '2025-12-16'], dtype=object)

In [4]:
vol = []
stock = []
for date in dates:
    daily_data = option_data[option_data['Date'] == date]
    S0 = daily_data['S0'].values[0]
    stock.append(S0)
    sigma = daily_data['sigma'].values[0]
    vol.append((sigma * np.sqrt(252))**2)
vol = np.array(vol)
stock = np.array(stock)

In [5]:
vol.size

25

In [6]:
ar_1 = ARIMA(vol, order=(1, 0, 0)).fit()
ar_1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                   25
Model:                 ARIMA(1, 0, 0)   Log Likelihood                  28.179
Date:                Sun, 01 Feb 2026   AIC                            -50.358
Time:                        20:07:20   BIC                            -46.701
Sample:                             0   HQIC                           -49.344
                                 - 25                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.4553      0.052     28.134      0.000       1.354       1.557
ar.L1          0.6685      0.256      2.609      0.009       0.166       1.171
sigma2         0.0060      0.001      4.120      0.000       0.003       0.009
===================================================================================
Ljung-Box (L1) (Q):                   0.18   Jarque-Bera (JB):                 7.84
Prob(Q):                              0.67   Prob(JB):                         0.02
Heteroskedasticity (H):               1.36   Skew:                             0.88
Prob(H) (two-sided):                  0.67   Kurtosis:                         5.10
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [7]:
theta, kappa, eta = ar_1.params[0], ar_1.params[1], vol.std()
theta, kappa, eta

(np.float64(1.4553067399234576),
 np.float64(0.6685270745047369),
 np.float64(0.10552900363772504))

In [8]:
stock.size

25

In [9]:
log_rets = np.log(stock[1:] / stock[:-1])
log_rets

array([ 0.00012946,  0.0011933 ,  0.00398279, -0.00398279,  0.00548857,
        0.00534026, -0.00474548, -0.00417663, -0.00288171,  0.01230576,
        0.00039107, -0.00048075, -0.00103859, -0.00549918, -0.0017763 ,
        0.00183584,  0.00584833, -0.00866402, -0.00466794, -0.00316488,
        0.00544172,  0.0057137 , -0.00075469, -0.00644475])

In [ ]:
vol_shocks = vol[1:] - vol[:-1]
vol_shocks

array([-0.00802361, -0.01480471, -0.11037858,  0.06261231, -0.04145496,
        0.03500929,  0.04947526,  0.02353798,  0.00876395,  0.25454392,
       -0.04397405, -0.06680739, -0.18212845,  0.04124358, -0.04253846,
        0.00452052,  0.06678185,  0.14039303, -0.04652768,  0.01730527,
        0.02081109, -0.02966082, -0.08561566,  0.09446005])

In [14]:
rho = np.corrcoef(log_rets, vol_shocks**2)[0, 1]
rho

np.float64(0.3479216959189683)

In [15]:
v0 = vol[0]
v0

np.float64(1.4078720746759057)

In [16]:
theta, kappa, eta, rho, v0

(np.float64(1.4553067399234576),
 np.float64(0.6685270745047369),
 np.float64(0.10552900363772504),
 np.float64(0.3479216959189683),
 np.float64(1.4078720746759057))